# PROJECT: MultiVelo v0.1.3 Analysis after building MultiVelo's velocity model using RNA, ATAC and LIAM data
---
**Title:** Predicting RNA-velocity using multiome data generated from iPSCs established for human muscle development.

**Description:** Downstream analysis of RNA velocity model from MultiVelo v0.1.3 `recover_dynamics_chrom` on multiome with alignment of all the objects, i.e., RNA, ATAC and LIAM.

**Author(s):** Dr. Shuba Varshini Alampalli

**Developer:** Implemented in CUBI's SODAR-Kiosc by Federico Marotta

**SODAR Data Steward:** Dr. Shuba Varshini Alampalli and Dr. Miha Milek

**Institute:** Core Unit Bioinformatics (CUBI), Berlin Institute of Health, Germany AND Charité Universitätsmedizin Berlin, Germany

**Contact:** cubi-helpdesk@bih-charite.de  

**Date:** 2026-07-22  
**Version:** 1.0  
**Environment:** `conda activate multivelo-0.1.3`

In [13]:
#!/usr/bin/env python
# coding: utf-8

# Standard Library
import math

# Data Handling & Statistics
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.stats as stats

# Single-Cell Genomics
import scanpy as sc
import scvelo as scv
import multivelo as mv

# Visualization
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colorbar as cbar
import seaborn as sns

# Jupyter Notebook & Interactive Widgets
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, clear_output

# --- Settings ---
scv.settings.verbosity = 3
sc.settings.verbosity = 3
scv.set_figure_params(style="scvelo", dpi=100)
plt.rcParams['pdf.fonttype'] = 42 # Ensures text is editable in Illustrator/Inkscape

# =========================================================
# THE FIX: MATPLOTLIB 3.6+ BUG PATCH
# =========================================================
if not hasattr(cbar.Colorbar, 'draw_all'):
    cbar.Colorbar.draw_all = lambda self: None

In [14]:
# =========================================================
# STAGE 1: SETUP & PREPROCESSING
# =========================================================
print("--- STAGE 1: LOADING SAVED ADATA_CLEAN OBJECT ---")

# Define the file path
file_path = "/sodarZone/projects/77/77044078-b82e-40c6-a5de-8e1a68e7fc4d/sample_data/study_e022ac77-81ef-47a9-882e-2d18d8d23254/assay_32d1d2b9-b432-4b7a-be83-5236889ef24c/MiscFiles/multivelo_result_hvg_5K_reorganising_filtering_normalisation_adata_clean_15July2026_processed.h5ad"
adata_clean = sc.read_h5ad(file_path)
print("\n--- DATA LOADED ---")

--- STAGE 1: LOADING SAVED ADATA_CLEAN OBJECT ---

--- DATA LOADED ---


In [15]:
print("Labelling data")

# 1. Biological Dictionaries
lineages = {
    "Neural lineage": ['3', '7', '8', '10', '14', '15'],
    "Myogenic lineage": ['0', '6', '11', '12', '16'],
    "Mesenchymal lineage": ['1', '2', '4', '5', '9'],
    "Other cells": ['17','20']
}

# Flatted strings for direct mapping
cluster_cell_type = {
    "0" : 'Dermomyotome-like cells',
    "1" : 'Multipotent mesenchymal progenitors 2',
    "2" : 'Mesenchymal chondrogenic progenitors 1',
    "3" : 'Dorsal and intermediary neural progenitors',
    "4" : 'Multipotent mesenchymal progenitors 1',
    "5" : 'Mesenchymal chondrogenic progenitors 2',
    "6" : 'Early myoprogenitors',
    "7" : 'Early differentiating dorsal and intermediary neurons',
    "8" : 'Maturing dorsal and intermediary neurons (glutamatergic + inhibitory)',
    "9" : 'Early FAPs',
    "10" : 'Dorsal neural progenitors',
    "11" : 'Multipotent myogenic-mesenchymal cells',
    "12" : 'Myonuclei',
    "14" : 'Astrocyte progenitors',
    "15" : 'Very early differentiating dorsal and intermediary neurons',
    "16" : 'Late myoprogenitors',
    "17" : 'Schwann cell progenitors',
    "20" : 'Endothelial cells'
}

cluster_cell_type_abbri = {
    "0" : 'DMyo',
    "1" : 'mMsP2',
    "2" : 'MsChP1',
    "3" : 'diNP',
    "4" : 'mMsP1',
    "5" : 'MsChP2',
    "6" : 'eMyo',
    "7" : 'ediNeu',
    "8" : 'diNeu',
    "9" : 'eFAP',
    "10" : 'dNP',
    "11" : 'mMyoMs',
    "12" : 'Myo',
    "14" : 'AP',
    "15" : 'eediNeu',
    "16" : 'lMyo',
    "17" : 'SCP',
    "20" : 'EC'
}


marker_genes_dict = {
    "0" : ['NCAM1', 'MEIS2', 'SOX5', 'PAX3', 'EYA1', 'SIX1', 'MEOX2', 'EYA2'],
    "1" : ['FTL', 'SOX5', 'CDH11', 'EBF2', 'ROR1', 'RORA', 'FOXP1', 'FOXD1', 'EBF1', 'FLI1', 'SOX9', 'RUNX1', 'PDGFRB', 'THY1', 'LOXL1'],
    "2" : ['SOX5', 'RORA', 'CDH11', 'ROR1', 'SOX6', 'POSTN', 'MEOX2', 'TENM2', 'SCX', 'ROR2', 'BARX2', 'SOX9'],
    "3" : ['CDH2', 'PAX3', 'GLI3', 'OLIG3', 'ZIC1', 'POU3F3', 'FABP7', 'HES5', 'GBX2', 'LMX1A', 'MSX1'],
    "4" : ['SOX5', 'RORA', 'EBF2', 'ROR1', 'CDH11', 'FTL', 'TWIST1', 'LOXL2', 'RUNX1', 'POSTN', 'FOXD1', 'RUNX2', 'MEOX1', 'THY1', 'SOX9', 'PDGFRB', 'LOX', 'BARX2', 'PRRX1', 'FOXC1'],
    "5" : ['SOX5', 'EBF2', 'SOX6', 'RORA', 'POSTN', 'SOX9', 'SCX', 'PRRX1', 'BARX2'],
    "6" : ['NCAM1', 'MEIS2', 'PAX3', 'SOX6', 'ERBB3', 'PBX3', 'MECOM', 'PAX7', 'MYOD1', 'CDH2', 'CDH15', 'MYOG'],
    "7" : ['NEUROD1', 'POU4F1', 'DRGX', 'ISL1', 'PAX2', 'LMX1B', 'ASCL1', 'INSM1', 'SLC6A5', 'SNAP2', 'SLC8A1', 'SLC4A7', 'BCL11A', 'BCL11B', 'TLX3', 'FOXD3', 'OLIG3'],
    "8" : ['TLX3', 'PAX2', 'NAV3', 'SYN3', 'SLC6A5', 'SNAP25', 'KCNQ5', 'LBX1'],
    "9" : ['RORA', 'ROR1', 'ROR2', 'PRRX1', 'LOX', 'MEIS2', 'TWIST1', 'PDGFRB', 'LOXL2', 'RUNX1', 'RUNX2', 'FOXD1', 'FOXC1', 'THY1', 'CDH2', 'PDGFRA', 'DCN', 'FN1', 'LUM', 'POSTN', 'FAP', 'VIM', 'COL1A1', 'COL1A2', 'COL3A1', 'PTN', 'OGN', 'FBLN5'],
    "10" : ['CDH2', 'PBX1', 'CDH6', 'ZIC1', 'MSX1', 'RSPO1', 'ZIC5', 'POU3F3', 'LMX1A', 'OLIG3', 'FABP7', 'PAX6'],
    "11" : ['DACH1', 'CDH11', 'MEIS2', 'RORA', 'ROR1', 'CDH2', 'TWIST1', 'EYA1', 'DACH2', 'MEOX1', 'MEOX2', 'SOX5', 'FBLN2', 'LOX', 'PAX7', 'LOXL2', 'EBF1', 'DPP4', 'PAX3', 'SIX1'],
    "12" : ['MYH3', 'MYL4', 'MYL11', 'NEB', 'ACTN2', 'DES', 'TTN', 'MYH7', 'MYH2', 'MYH8'],
    "14" : ['VIM', 'SLC1A3', 'ALDH1L1', 'SOX9', 'FGFR3'],
    "15" : ['ASCL1', 'TLX3', 'SLC17A6', 'OLIG3', 'FOXD3', 'GDF7', 'HES5', 'SLC17A6', 'LMX1A', 'LHX2', 'LMX1B', 'PAX3'],
    "16" : ['TUBA1A', 'TNNT1', 'DCX', 'PAX7', 'PITX2', 'SOX5', 'PBX3', 'DES', 'TBX3', 'MYOD', 'MYF5'],
    "17" : ['PLP1', 'CDH6', 'CDH19', 'ERBB3', 'SOX10'],
    "20" : ['TEK', 'CDH5', 'FLT4', 'CDH19']
}

# Clean marker dictionary for available genes
valid_markers = {cl: [g for g in genes if g in adata_clean.var_names] for cl, genes in marker_genes_dict.items() if any(g in adata_clean.var_names for g in genes)}

# 2. Metadata Formatting & Mapping

# Convert to string to safely replace the text without triggering categorical errors
adata_clean.obs['day'] = adata_clean.obs['day'].astype(str)
adata_clean.obs['day'] = adata_clean.obs['day'].replace({'D22-15': 'Endpoint'})

# Enforce Day ordering safely using the new label
day_order = ['D8', 'D12', 'D20', 'Endpoint']

# Convert back to category and apply the strict chronological order
adata_clean.obs['day'] = adata_clean.obs['day'].astype('category')
adata_clean.obs['day'] = adata_clean.obs['day'].cat.set_categories(day_order, ordered=True)

# Macro lineage columns treated as categories
adata_clean.obs['day'] = adata_clean.obs['day'].astype('category')
adata_clean.obs['Macro_Lineage'] = adata_clean.obs['Macro_Lineage'].astype('category')

# Map Cell Types
adata_clean.obs['cell_type'] = adata_clean.obs['leiden_merged'].astype(str).map(cluster_cell_type).fillna('Unknown').astype('category')
adata_clean.obs['cell_type_abbri'] = adata_clean.obs['leiden_merged'].astype(str).map(cluster_cell_type_abbri).fillna('Unknown').astype('category')

# Create descriptive Spatio-Temporal column 
raw_day_cell_type = (
    adata_clean.obs['day'].astype(str) + "_C" + 
    adata_clean.obs['leiden_merged'].astype(str) + "_" + 
    adata_clean.obs['cell_type'].astype(str)
)

# Sort the new categories chronologically based on day_order
sorted_categories = sorted(
    raw_day_cell_type.unique(), 
    key=lambda x: day_order.index(x.split('_')[0]) if x.split('_')[0] in day_order else 999
)
adata_clean.obs['day_cell_type'] = pd.Categorical(raw_day_cell_type, categories=sorted_categories, ordered=True)

# Map Macro Lineages
cluster_to_lineage = {str(c): lin for lin, clusters in lineages.items() for c in clusters}
adata_clean.obs['lineage'] = adata_clean.obs['leiden_merged'].astype(str).map(cluster_to_lineage).astype('category')

def assign_macro_lineage(cluster):
    if str(cluster) in lineages['Myogenic lineage']: return 'Myogenic lineage'
    elif str(cluster) in lineages['Neural lineage']: return 'Neural lineage'
    elif str(cluster) in lineages['Mesenchymal lineage']: return 'Mesenchymal lineage'
    return 'Other cells'
    
adata_clean.obs['Macro_Lineage'] = adata_clean.obs['leiden_merged'].apply(assign_macro_lineage).astype('category')

# Assign standard Scanpy palettes to them
if 'day_colors' not in adata_clean.uns:
    import scanpy.plotting._utils as sc_utils
    sc_utils.add_colors_for_categorical_sample_annotation(adata_clean, 'day')

if 'Macro_Lineage_colors' not in adata_clean.uns:
    import scanpy.plotting._utils as sc_utils
    sc_utils.add_colors_for_categorical_sample_annotation(adata_clean, 'Macro_Lineage')

print("Labelling data done!")

Labelling data
Labelling data done!


In [16]:
print("\n--- GLOBAL OVERVIEW (VELOCITY & LATENT TIME) ---")

def plot_interactive_velocity(plot_choice):
    # Clear the current figure so plots don't overlap when you switch options
    plt.clf() 
    
    if plot_choice == 'Cell Type Trajectories':
        mv.velocity_embedding_stream(
            adata_clean, 
            basis='umap', 
            color='cell_type', 
            title='Differentiation Trajectories', 
            legend_loc='right margin', 
            alpha=0.8, 
            linewidth=1.5, 
            show=False
        )
        
    elif plot_choice == 'Cell Type Trajectories (Abbreviated)':
        mv.velocity_embedding_stream(
            adata_clean, 
            basis='umap', 
            color='cell_type_abbri', 
            title='Differentiation Trajectories', 
            legend_loc='right margin', 
            alpha=0.8, 
            linewidth=1.5, 
            show=False
        )
        
    elif plot_choice == 'Temporal Progression (Day)':
        mv.velocity_embedding_stream(
            adata_clean, 
            basis='umap', 
            color='day', 
            title='Temporal Progression (Streamlines)', 
            legend_loc='right margin', 
            alpha=0.8, 
            linewidth=1.5, 
            show=False
        )
        
    elif plot_choice == 'Global Latent Time':
        sc.pl.umap(
            adata_clean, 
            color='latent_time', 
            cmap='gnuplot', 
            title='Latent Time (Inferred Maturation)', 
            size=20,
            sort_order=True,
            show=False,
            frameon=False # Removes the harsh box around the plot for a cleaner look
        )
    
    # Display the selected plot
    plt.show()

# Define the dropdown options
plot_options = [
    'Cell Type Trajectories',
    'Cell Type Trajectories (Abbreviated)',
    'Temporal Progression (Day)',
    'Global Latent Time'
]

# Create the interactive widget
interact(plot_interactive_velocity, plot_choice=widgets.Dropdown(
    options=plot_options,
    value='Cell Type Trajectories',
    description='Select Plot:',
    disabled=False
))

print("\n--- Done ---")


--- GLOBAL OVERVIEW (VELOCITY & LATENT TIME) ---


interactive(children=(Dropdown(description='Select Plot:', options=('Cell Type Trajectories', 'Cell Type Traje…


--- Done ---


In [17]:
print("\n--- GENERATING INTERACTIVE DAY-SPLIT TRAJECTORIES & LATENT TIME ---")

# 1. Get the list of valid days from the dataset
days = list(adata_clean.obs["day"].cat.categories)

# 2. Create the dropdown widget for Day selection
day_dropdown = widgets.Dropdown(
    options=days,
    description='Select Day:',
    style={'description_width': 'initial'}
)

# 3. The Interactive Plotting Function
@interact(day=day_dropdown)
def plot_interactive_day_split(day):
    print(f"Generating Trajectories and Latent Time for Day {day}...")
    
    # Subset data for the selected day
    adata_day = adata_clean[adata_clean.obs["day"] == day].copy()
    
    # Check if there are enough cells to plot streamlines properly
    if adata_day.n_obs < 10:
        print(f"  [!] Skipping Day {day} - too few cells for valid streamlines.")
        return

    # Create a 1x2 subplot layout for side-by-side comparison
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    # --- PLOT 1: Velocity Streamlines (On-data labels) ---
    mv.velocity_embedding_stream(
        adata_day,
        basis="umap",
        color="cell_type_abbri",
        title=f"Lineage Trajectories: Day {day}",
        legend_loc='on data', 
        legend_fontoutline=2, # Makes labels readable over dots/arrows
        alpha=0.8,
        linewidth=1.5,
        frameon=False,
        show=False,
        ax=axes[0]
    )
    
    # --- PLOT 2: Latent Time (Independent Scaling) ---
    # We use sc.pl.umap to easily handle the independent colorbar per day
    sc.pl.umap(
        adata_day,
        color="latent_time",
        cmap="gnuplot",
        size=35,
        title=f"Latent Time: Day {day}",
        ax=axes[1],
        show=False,
        frameon=False,
        colorbar_loc='right' 
    )
    
    # Adjust layout to prevent overlap between plots
    plt.tight_layout()
    
    # Display the plots
    plt.show()
    
    # Free memory
    plt.close(fig)

print("\n--- Done ---")


--- GENERATING INTERACTIVE DAY-SPLIT TRAJECTORIES & LATENT TIME ---


interactive(children=(Dropdown(description='Select Day:', options=('D8', 'D12', 'D20', 'Endpoint'), style=Desc…


--- Done ---


In [18]:
print("\n--- Generating ATAC AND Ms UMAP AND PHASE PORTRAITS FOR INDIVIDUAL GENES---")

# Matplotlib patch to prevent the scvelo crash
if not hasattr(mpl.colorbar.Colorbar, 'draw_all'):
    mpl.colorbar.Colorbar.draw_all = lambda self: None

# Extract ALL genes available in your AnnData object
all_genes = adata_clean.var_names.tolist()
# Optional: Sort them alphabetically so they are easier to find in the dropdown
all_genes = sorted(all_genes)

# 1. Create the Dropdown Widget
gene_dropdown = widgets.Dropdown(
    options=all_genes,
    value=all_genes[0] if all_genes else None,
    description='Target Gene:',
    disabled=False,
)

# 2. Create an Output area where the plot will render
plot_output = widgets.Output()

# 3. Define the function that runs when a new gene is selected
def update_umap_plots(change):
    with plot_output:
        clear_output(wait=True) # Clears the old plot to prevent stacking
        gene = change['new']
        
        print(f"Generating ATAC and Ms UMAPs for {gene}...")
        
        # Calculate the maximums dynamically for the CURRENTLY selected gene
        max_atac = np.nanmax(adata_clean.obs_vector(gene, layer='ATAC'))
        max_spliced = np.nanmax(adata_clean.obs_vector(gene, layer='Ms'))
        
        # Create a 1x2 subplot for ATAC and Ms
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # =========================================================
        # ATAC LAYER UMAP
        # =========================================================
        scv.pl.scatter(
            adata_clean, 
            basis='umap', 
            color=gene, 
            layer='ATAC', 
            color_map='Greens',        
            vmin=0, 
            vmax=max_atac, 
            title=f"{gene}\nChromatin Accessibility (ATAC)",
            ax=axes[0],
            show=False
        )
        
        # =========================================================
        # Ms LAYER UMAP
        # =========================================================
        scv.pl.scatter(
            adata_clean, 
            basis='umap', 
            color=gene, 
            layer='Ms', 
            color_map='Reds',          
            vmin=0, 
            vmax=max_spliced, 
            title=f"{gene}\nSmoothed RNA (Ms)",
            ax=axes[1],
            show=False
        )
        
        mv.scatter_plot(adata_clean, gene, color_by='day',by='cus')
        plt.title(f"{gene} (Colored by Day)")
        plt.show()
        
        mv.scatter_plot(adata_clean, gene, color_by='Macro_Lineage',by='cus')
        plt.title(f"{gene} (Colored by Lineage)")
        plt.show()

        mv.scatter_plot(adata_clean, gene, color_by='latent_time',by='cus')
        plt.title(f"{gene} (Colored by Latent time)")
        plt.show()
        
        plt.tight_layout()
        plt.show()

# 4. Bind the function to the dropdown and display
gene_dropdown.observe(update_umap_plots, names='value')

# Display the UI
display(widgets.VBox([
    widgets.HTML("<h3>Interactive ATAC & Ms UMAP Explorer</h3>"),
    gene_dropdown, 
    plot_output
]))

# Manually trigger the first plot initialization
if gene_dropdown.value is not None:
    update_umap_plots({'new': gene_dropdown.value})


--- Generating ATAC AND Ms UMAP AND PHASE PORTRAITS FOR INDIVIDUAL GENES---


In [19]:
print("\n--- GENERATING INTERACTIVE PHASE PORTRAIT OF MARKER-GENES ---")

# List of categorical columns you plan to use for coloring
categorical_columns = ['day', 'Macro_Lineage']

for col in categorical_columns:
    if col in adata_clean.obs.columns:
        # 1. Ensure it is a categorical variable
        adata_clean.obs[col] = adata_clean.obs[col].astype('category')
        
        # 2. Force Scanpy to assign a color palette
        if f'{col}_colors' not in adata_clean.uns:
            # A quick dummy plot to initialize colors quietly
            sc.pl.umap(adata_clean, color=col, show=False)
            
#sc.pl.umap(adata_clean, color='day', show=False)
#sc.pl.umap(adata_clean, color='Macro_Lineage', show=False)

# Ensure we have our list of valid genes (from your existing script)
target_genes = list(set([gene for gene_list in valid_markers.values() for gene in gene_list]))
valid_genes_to_plot = [g for g in target_genes if g in adata_clean.var_names]

# 1. Create the Dropdown Widget
gene_dropdown = widgets.Dropdown(
    options=valid_genes_to_plot,
    value=valid_genes_to_plot[0] if valid_genes_to_plot else None,
    description='Target Gene:',
    disabled=False,
)

# 2. Create an Output area where the plot will render
plot_output = widgets.Output()

# 3. Define the function that runs when a new gene is selected
def update_phase_portrait(change):
    with plot_output:
        clear_output(wait=True) # Clears the old plot
        gene = change['new']
        
        # Create a 1x3 subplot for Day, Lineage, and Latent Time
        print(f"Generating Phase Portraits for {gene}...")
        
        # We plot them sequentially just like your PDF loop, but to the screen
        mv.scatter_plot(adata_clean, gene, color_by='day',by='cus')
        plt.title(f"{gene} (Colored by Day)")
        plt.show()
        
        mv.scatter_plot(adata_clean, gene, color_by='Macro_Lineage',by='cus')
        plt.title(f"{gene} (Colored by Lineage)")
        plt.show()

# 4. Bind the function to the dropdown and display
gene_dropdown.observe(update_phase_portrait, names='value')

# Display the UI
display(widgets.VBox([
    widgets.HTML("<h3>Interactive Phase Portrait Explorer</h3>"),
    gene_dropdown, 
    plot_output
]))

# Manually trigger the first plot
update_phase_portrait({'new': gene_dropdown.value})

print("\n--- Done ---")


--- GENERATING INTERACTIVE PHASE PORTRAIT OF MARKER-GENES ---



--- Done ---


In [20]:
# Configuration (Adjust these if needed)
FDR_THRESH = 0.05
TOP_N_BROAD = 100    # For Lineage and Day
TOP_N_SPATIO = 50    # For Day_Cell_Type
MAX_SHARES = 3       # Intra-lineage sharing limit for spatio-temporal drivers
# =========================================================
# SIGNIFICANCE & SUBTRACTION LOGIC
# =========================================================
print("\n--- EXTRACTING SIGNIFICANT UNIQUE DRIVERS ---")

def get_sig_genes(rank_key, group, top_n):
    """Fetches top N genes, heavily filtered by Statistical Significance (FDR)."""
    df = sc.get.rank_genes_groups_df(adata_clean, group=group, key=rank_key)
    sig_df = df[df['pvals_adj'] < FDR_THRESH]
    return sig_df['names'].head(top_n).tolist()

def get_mutually_exclusive(rank_key, obs_col, top_n):
    """Calculates completely unique genes for broad categories (Lineage/Day)."""
    categories = adata_clean.obs[obs_col].cat.categories
    sig_dict = {cat: set(get_sig_genes(rank_key, cat, top_n)) for cat in categories}
    
    unique_dict = {}
    for cat in categories:
        # Subtract all genes found in any other category
        other_genes = set.union(*(sig_dict[other] for other in categories if other != cat))
        unique_dict[cat] = list(sig_dict[cat] - other_genes)
    return unique_dict

def get_spatio_temporal_drivers(rank_key, mapping_dict):
    """Applies the Iron Wall logic to high-resolution micro-states."""
    groups = adata_clean.obs['day_cell_type'].cat.categories
    sig_dict = {g: get_sig_genes(rank_key, g, TOP_N_SPATIO) for g in groups if g in mapping_dict}
    
    # Map where each gene appears
    gene_locs = {}
    for g, genes in sig_dict.items():
        for gene in genes:
            gene_locs.setdefault(gene, set()).add(g)
            
    unique_dict = {}
    for g, genes in sig_dict.items():
        target_lin = mapping_dict[g]
        valid = []
        for gene in genes:
            locs = gene_locs[gene]
            # Rule 1: No cross-lineage bleed
            if any(mapping_dict.get(loc) != target_lin for loc in locs): continue
            # Rule 2: Limit intra-lineage sharing
            if len(locs) <= MAX_SHARES:
                valid.append(gene)
        if valid:
            unique_dict[g] = valid
    return unique_dict

# Extract mapping dictionary for Spatio-Temporal filter
mapping_df = adata_clean.obs[['day_cell_type', 'lineage']].drop_duplicates().dropna()
group_to_lin = dict(zip(mapping_df['day_cell_type'], mapping_df['lineage']))

# 1. Broad Lineage Extraction
unique_lin_atac = get_mutually_exclusive('rank_atac_lineage', 'lineage', TOP_N_BROAD)
unique_lin_vel = get_mutually_exclusive('rank_vel_lineage', 'lineage', TOP_N_BROAD)

# 2. Broad Day Extraction
unique_day_atac = get_mutually_exclusive('rank_atac_day', 'day', TOP_N_BROAD)
unique_day_vel = get_mutually_exclusive('rank_vel_day', 'day', TOP_N_BROAD)

# 3. Spatio-Temporal Extraction
unique_spatio_atac = get_spatio_temporal_drivers('rank_atac_day_cluster', group_to_lin)
unique_spatio_vel = get_spatio_temporal_drivers('rank_vel_day_cluster', group_to_lin)


# =========================================================
# COMPILATION & CSV EXPORT (Union & Difference)
# =========================================================
print("\n--- COMPILING STATISTICAL MASTERSHEET ---")

# Pre-load all statistical tables and index them for lightning-fast lookups
master_stats = {
    'Lin_ATAC': sc.get.rank_genes_groups_df(adata_clean, group=None, key='rank_atac_lineage').set_index(['group', 'names']),
    'Lin_Vel': sc.get.rank_genes_groups_df(adata_clean, group=None, key='rank_vel_lineage').set_index(['group', 'names']),
    'Day_ATAC': sc.get.rank_genes_groups_df(adata_clean, group=None, key='rank_atac_day').set_index(['group', 'names']),
    'Day_Vel': sc.get.rank_genes_groups_df(adata_clean, group=None, key='rank_vel_day').set_index(['group', 'names']),
    'Spatio_ATAC': sc.get.rank_genes_groups_df(adata_clean, group=None, key='rank_atac_day_cluster').set_index(['group', 'names']),
    'Spatio_Vel': sc.get.rank_genes_groups_df(adata_clean, group=None, key='rank_vel_day_cluster').set_index(['group', 'names']),
}

def safe_stat_lookup(df_key, group, gene):
    """Safely retrieves P-val, FDR, and LogFC if they exist."""
    try:
        data = master_stats[df_key].loc[(group, gene)]
        return data['pvals'], data['pvals_adj'], data['logfoldchanges']
    except KeyError:
        return np.nan, np.nan, np.nan

csv_rows = []

def build_export_rows(atac_dict, vel_dict, level_name, atac_stat_key, vel_stat_key):
    """Combines ATAC and Vel lists, determines the difference, and fetches stats."""
    all_groups = set(atac_dict.keys()) | set(vel_dict.keys())
    
    for group in all_groups:
        atac_genes = set(atac_dict.get(group, []))
        vel_genes = set(vel_dict.get(group, []))
        combined_genes = atac_genes | vel_genes  # The Union (Combined List)
        
        for gene in combined_genes:
            # Determine Difference/Intersection status
            if gene in atac_genes and gene in vel_genes:
                presence = "Both (Intersection)"
            elif gene in atac_genes:
                presence = "ATAC Only (Difference)"
            else:
                presence = "Velocity Only (Difference)"
                
            # Fetch stats
            a_pval, a_fdr, a_lfc = safe_stat_lookup(atac_stat_key, group, gene)
            v_pval, v_fdr, v_lfc = safe_stat_lookup(vel_stat_key, group, gene)
            
            csv_rows.append({
                'Extraction_Level': level_name,
                'Specific_Group': group,
                'Gene': gene,
                'Modality_Presence': presence,
                'ATAC_FDR': a_fdr,
                'ATAC_LogFC': a_lfc,
                'Velocity_FDR': v_fdr,
                'Velocity_LogFC': v_lfc
            })

# Compile the rows
build_export_rows(unique_lin_atac, unique_lin_vel, '1. Macro-Lineage', 'Lin_ATAC', 'Lin_Vel')
build_export_rows(unique_day_atac, unique_day_vel, '2. Culture Day', 'Day_ATAC', 'Day_Vel')
build_export_rows(unique_spatio_atac, unique_spatio_vel, '3. Spatio-Temporal', 'Spatio_ATAC', 'Spatio_Vel')

# Finalize and Save
df_master = pd.DataFrame(csv_rows)

# Sort it logically so it looks beautiful in Excel
df_master = df_master.sort_values(by=['Extraction_Level', 'Specific_Group', 'Modality_Presence'])

print("\n--- Done ---")


--- EXTRACTING SIGNIFICANT UNIQUE DRIVERS ---

--- COMPILING STATISTICAL MASTERSHEET ---

--- Done ---


In [21]:
# =========================================================
# INTERACTIVE BROAD MARKERS: DUAL-OMIC HEATMAPS
# =========================================================
print("\n--- GENERATING INTERACTIVE BROAD DUAL-OMIC HEATMAPS ---")

def build_criteria_dicts(atac_dict, vel_dict, top_n=10):
    """Helper function to sort genes into 3 criteria (Both, ATAC-only, Vel-only)"""
    dict_intersection = {}
    dict_atac_only = {}
    dict_vel_only = {}

    all_groups = set(atac_dict.keys()) | set(vel_dict.keys())

    for group in all_groups:
        a_genes = atac_dict.get(group, [])
        v_genes = vel_dict.get(group, [])
        
        # Criteria 1: Common to BOTH ATAC and Velocity
        common = [g for g in a_genes if g in v_genes][:top_n]
        if common: dict_intersection[group] = common
        
        # Criteria 2: Difference (ATAC Only)
        a_only = [g for g in a_genes if g not in v_genes][:top_n]
        if a_only: dict_atac_only[group] = a_only
            
        # Criteria 3: Difference (Velocity Only)
        v_only = [g for g in v_genes if g not in a_genes][:top_n]
        if v_only: dict_vel_only[group] = v_only
            
    return dict_intersection, dict_atac_only, dict_vel_only

# 1. Pre-build all the dictionaries for both Lineage and Day (Top 10 genes)
lin_intersection, lin_atac_only, lin_vel_only = build_criteria_dicts(unique_lin_atac, unique_lin_vel, top_n=10)
day_intersection, day_atac_only, day_vel_only = build_criteria_dicts(unique_day_atac, unique_day_vel, top_n=10)

# 2. Map the UI choices to the correct dictionary
heatmap_data_map = {
    'Macro_Lineage': {
        'Intersection (ATAC & Vel)': lin_intersection,
        'ATAC Unique': lin_atac_only,
        'Velocity Unique': lin_vel_only
    },
    'day': {
        'Intersection (ATAC & Vel)': day_intersection,
        'ATAC Unique': day_atac_only,
        'Velocity Unique': day_vel_only
    }
}

# 3. Create the Dropdown Widgets
group_dropdown = widgets.Dropdown(
    options=['Macro_Lineage', 'day'],
    description='Group By:'
)

criteria_dropdown = widgets.Dropdown(
    options=['Intersection (ATAC & Vel)', 'ATAC Unique', 'Velocity Unique'],
    description='Criteria:',
    style={'description_width': 'initial'}
)

# 4. The Interactive Function
@interact(groupby_col=group_dropdown, criteria=criteria_dropdown)
def plot_interactive_broad_heatmap(groupby_col, criteria):
    
    gene_dict = heatmap_data_map[groupby_col][criteria]
    title = f"{groupby_col.replace('_', ' ').title()} Drivers: {criteria}"
    
    target_genes = []
    for genes in gene_dict.values():
        target_genes.extend([g for g in genes if g in adata_clean.var_names])
    
    target_genes = list(dict.fromkeys(target_genes))
    if not target_genes:
        print(f"  [!] No valid genes found for {title}.")
        return

    # Extract means grouped by our target column (Macro_Lineage or day)
    df_a = sc.get.obs_df(adata_clean, keys=[groupby_col] + target_genes, layer='ATAC').groupby(groupby_col).mean()
    df_v = sc.get.obs_df(adata_clean, keys=[groupby_col] + target_genes, layer='velocity').groupby(groupby_col).mean()
    
    # Scale 0-1 independently per gene
    df_a = (df_a - df_a.min(axis=0)) / (df_a.max(axis=0) - df_a.min(axis=0)).replace(0, 1)
    df_v = (df_v - df_v.min(axis=0)) / (df_v.max(axis=0) - df_v.min(axis=0)).replace(0, 1)
    
    # Dynamic Sizing based on gene count
    fig_width = max(18, len(target_genes) * 0.4)
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(fig_width, 6), gridspec_kw={'wspace': 0.05})
    
    sns.heatmap(df_a, cmap='Greens', ax=axes[0], cbar_kws={'shrink': 0.8}, xticklabels=True)
    axes[0].set_title('DNA State (ATAC)', fontsize=14, pad=10)
    axes[0].set_ylabel(groupby_col.replace('_', ' ').title(), fontsize=12)
    axes[0].tick_params(axis='x', rotation=90, labelsize=9)
    
    sns.heatmap(df_v, cmap='Reds', ax=axes[1], cbar_kws={'shrink': 0.8}, xticklabels=True)
    axes[1].set_title('RNA Engines (Velocity)', fontsize=14, pad=10)
    axes[1].set_ylabel('')
    axes[1].set_yticks([]) # Fuse the plots visually
    axes[1].tick_params(axis='x', rotation=90, labelsize=9)
    
    plt.suptitle(title, fontsize=18, fontweight='bold', y=1.05)
    
    plt.show()
    plt.close(fig)

print("\n--- Done ---")


--- GENERATING INTERACTIVE BROAD DUAL-OMIC HEATMAPS ---


interactive(children=(Dropdown(description='Group By:', options=('Macro_Lineage', 'day'), value='Macro_Lineage…


--- Done ---


In [22]:
# =========================================================
# INTERACTIVE SPATIO-TEMPORAL MARKERS (DUAL-OMIC)
# =========================================================
print("\n--- GENERATING INTERACTIVE SPATIO-TEMPORAL DUAL HEATMAPS ---")

# 1. Build the Three Criteria Dictionaries
dict_intersection = {}
dict_atac_only = {}
dict_vel_only = {}

# Get all unique day_cell_type micro-states that survived our filters
all_microstates = set(unique_spatio_atac.keys()) | set(unique_spatio_vel.keys())

# Limit to top 3 genes per state to prevent the heatmap x-axis from becoming unreadable
TOP_N_PLOT = 3 

for state in all_microstates:
    a_genes = unique_spatio_atac.get(state, [])
    v_genes = unique_spatio_vel.get(state, [])
    
    # Criteria 1: Common to BOTH ATAC and Velocity
    common = [g for g in a_genes if g in v_genes][:TOP_N_PLOT]
    if common: dict_intersection[state] = common
    
    # Criteria 2: Difference (ATAC Only)
    a_only = [g for g in a_genes if g not in v_genes][:TOP_N_PLOT]
    if a_only: dict_atac_only[state] = a_only
        
    # Criteria 3: Difference (Velocity Only)
    v_only = [g for g in v_genes if g not in a_genes][:TOP_N_PLOT]
    if v_only: dict_vel_only[state] = v_only

# 2. Map descriptive names to the generated dictionaries
heatmap_categories = {
    "Intersection (ATAC & Vel)": dict_intersection,
    "ATAC Unique": dict_atac_only,
    "Velocity Unique": dict_vel_only
}

# 3. Create the Dropdown Widget
category_dropdown = widgets.Dropdown(
    options=list(heatmap_categories.keys()),
    description='Criteria:',
    style={'description_width': 'initial'}
)

# 4. The Interactive Function
@interact(category=category_dropdown)
def plot_spatio_dual_heatmap(category):
    """Extracts, scales, and stitches ATAC & Vel side-by-side for the selected dictionary."""
    
    gene_dict = heatmap_categories[category]
    title = f"Spatio-Temporal Drivers: {category}"
    
    # Flatten dict and verify genes exist in the clean object
    target_genes = []
    for genes in gene_dict.values():
        target_genes.extend([g for g in genes if g in adata_clean.var_names])
    
    # Remove duplicates but preserve order
    target_genes = list(dict.fromkeys(target_genes))
    if not target_genes:
        print(f"  [!] No valid genes found for {title}.")
        return

    # Extract means grouped by day_cell_type
    df_a = sc.get.obs_df(adata_clean, keys=['day_cell_type'] + target_genes, layer='ATAC').groupby('day_cell_type').mean()
    df_v = sc.get.obs_df(adata_clean, keys=['day_cell_type'] + target_genes, layer='velocity').groupby('day_cell_type').mean()
    
    # Scale 0-1 independently per gene
    df_a = (df_a - df_a.min(axis=0)) / (df_a.max(axis=0) - df_a.min(axis=0)).replace(0, 1)
    df_v = (df_v - df_v.min(axis=0)) / (df_v.max(axis=0) - df_v.min(axis=0)).replace(0, 1)
    
    # Dynamic Sizing based on gene count
    # Grants at least 0.4 inches per gene, with a minimum width of 24
    fig_width = max(24, len(target_genes) * 0.4)
    
    # Plotting using the dynamic width
    fig, axes = plt.subplots(1, 2, figsize=(fig_width, 12), gridspec_kw={'wspace': 0.02})
    
    sns.heatmap(df_a, cmap='Greens', ax=axes[0], cbar_kws={'shrink': 0.8}, xticklabels=True)
    axes[0].set_title('DNA State (ATAC)', fontsize=14, pad=10)
    axes[0].set_ylabel('Spatio-Temporal Micro-State (day_cell_type)', fontsize=12)
    axes[0].tick_params(axis='x', rotation=90, labelsize=9)
    
    sns.heatmap(df_v, cmap='Reds', ax=axes[1], cbar_kws={'shrink': 0.8}, xticklabels=True)
    axes[1].set_title('RNA Engines (Velocity)', fontsize=14, pad=10)
    axes[1].set_ylabel('')
    axes[1].set_yticks([]) # Fuse the plots visually
    axes[1].tick_params(axis='x', rotation=90, labelsize=9)
    
    plt.suptitle(title, fontsize=18, fontweight='bold', y=1.02)
    
    # Show the plot to the notebook output
    plt.show()
    
    # Close the figure to free up memory
    plt.close(fig)

print("\n--- Done ---")


--- GENERATING INTERACTIVE SPATIO-TEMPORAL DUAL HEATMAPS ---


interactive(children=(Dropdown(description='Criteria:', options=('Intersection (ATAC & Vel)', 'ATAC Unique', '…


--- Done ---


In [23]:
print("\n--- GENERATING LATENT TIME DRIVERS PER CELL TYPE ---")

# 1. Setup UI Elements
cell_types = sorted(adata_clean.obs['cell_type'].dropna().unique())

ct_dropdown = widgets.Dropdown(
    options=cell_types,
    description='Cell Type:',
)
heatmap_output = widgets.Output()

# 2. The update function containing your math from Cell 16
def update_heatmap(change):
    with heatmap_output:
        clear_output(wait=True)
        ct = change['new']
        print(f"Calculating Latent Time correlations for {ct}...")
        
        mask_ct = (adata_clean.obs['cell_type'] == ct) & adata_clean.obs['latent_time'].notna()
        adata_ct = adata_clean[mask_ct].copy()
        
        if adata_ct.n_obs < 15:
            print(f"Skipping {ct} - too few cells ({adata_ct.n_obs}).")
            return
            
        # Chronological day sort logic
        day_weights = {'D8': 0, 'D12': 1, 'D20': 2, 'Endpoint': 3}
        adata_ct.obs['day_chrono_sort'] = adata_ct.obs['day'].map(day_weights).astype(float) + \
            (adata_ct.obs['latent_time'] / adata_ct.obs['latent_time'].max()) * 0.99
            
        # Spearman Correlation Logic
        expr_matrix = adata_ct.X.toarray() if sp.issparse(adata_ct.X) else adata_ct.X
        latent_times = adata_ct.obs['latent_time'].values
        
        correlations = [
            stats.spearmanr(expr_matrix[:, i], latent_times)[0] if np.std(expr_matrix[:, i]) > 0 else 0 
            for i in range(expr_matrix.shape[1])
        ]
            
        corr_df = pd.DataFrame({'Gene': adata_ct.var_names, 'Spearman_Corr': correlations}).set_index('Gene')
        top_late = corr_df.sort_values('Spearman_Corr', ascending=False).head(10).index.tolist()
        top_early = corr_df.sort_values('Spearman_Corr', ascending=True).head(10).index.tolist()
        
        # Plotting
        scv.pl.heatmap(
            adata_ct, 
            var_names=top_early + top_late, 
            sortby='day_chrono_sort', 
            col_color='day', 
            n_convolve=30,  
            figsize=(12, 5), 
            font_scale=0.7,  
            show=False
        )
        plt.suptitle(f'{ct} Maturation Dynamics', y=1.08, fontsize=16, fontweight='bold')
        plt.show()

# 3. Bind and Display
ct_dropdown.observe(update_heatmap, names='value')
display(widgets.VBox([
    widgets.HTML("<h3>Dynamic Maturation Heatmap Explorer</h3>"),
    ct_dropdown, 
    heatmap_output
]))

update_heatmap({'new': ct_dropdown.value})

print("\n--- Done ---")


--- GENERATING LATENT TIME DRIVERS PER CELL TYPE ---



--- Done ---


In [24]:
# =========================================================
# INTERACTIVE: LINEAGE-SPECIFIC LATENT TIME HEATMAPS
# =========================================================
print("\n--- INTERACTIVE: LATENT TIME DRIVERS PER MACRO-LINEAGE ---")

# 1. Identify all unique broad lineages with valid latent time data
macro_lineages = adata_clean.obs['Macro_Lineage'].dropna().unique()
macro_lineages = sorted(macro_lineages)

def plot_interactive_lineage_heatmap(lin):
    # Clear any existing plots to prevent overlapping
    plt.clf() 
    
    print(f"-> Processing Broad Lineage: {lin}...")
    
    # Isolate all cells belonging to this macro-lineage that have a valid clock
    mask_lin = (adata_clean.obs['Macro_Lineage'] == lin) & adata_clean.obs['latent_time'].notna()
    adata_lin = adata_clean[mask_lin].copy()
    
    # Safety Check: Skip if the lineage is too small to build a proper distribution
    if adata_lin.n_obs < 20:
        print(f"     [!] Skipping {lin} - too few cells ({adata_lin.n_obs}) for dynamic trajectory.")
        return
        
    # Chronologically arrange the cell rows perfectly along the lineage's developmental clock
    adata_lin = adata_lin[adata_lin.obs.sort_values('latent_time').index].copy()
    
    # 2. Compute Spearman Correlation for all genes against Latent Time across the whole lineage
    # Note: Depending on your dataset size, calculating this on-the-fly may take a few seconds per click.
    expr_matrix = adata_lin.X.toarray() if sp.issparse(adata_lin.X) else adata_lin.X
    latent_times = adata_lin.obs['latent_time'].values
    
    correlations = []
    for i in range(expr_matrix.shape[1]):
        gene_expr = expr_matrix[:, i]
        if np.std(gene_expr) == 0:
            correlations.append(0)
        else:
            corr, _ = stats.spearmanr(gene_expr, latent_times)
            correlations.append(corr)
            
    # Wrap results into our dataframe
    corr_df = pd.DataFrame({
        'Gene': adata_lin.var_names,
        'Spearman_Corr': correlations
    }).set_index('Gene')
    
    # Split into 20 Early (suppressed over time) and 20 Late (activated over time) lineage drivers
    top_late_genes = corr_df.sort_values('Spearman_Corr', ascending=False).head(20).index.tolist()
    top_early_genes = corr_df.sort_values('Spearman_Corr', ascending=True).head(20).index.tolist()
    
    heatmap_genes = top_early_genes + top_late_genes
    
    # --- SORTING LOGIC (Day -> Latent Time) ---
    day_weights = {'D8': 0, 'D12': 1, 'D20': 2, 'Endpoint': 3}
    
    # Primary sort = Day (0, 1, 2, 3), Secondary sort = normalized latent time (0 to 0.99)
    adata_lin.obs['day_chrono_sort'] = adata_lin.obs['day'].map(day_weights).astype(float) + \
        (adata_lin.obs['latent_time'] / adata_lin.obs['latent_time'].max()) * 0.99
    
    # 3. Visualize using scVelo's sliding-window smoothed heatmap
    try:
        scv.pl.heatmap(
            adata_lin, 
            var_names=heatmap_genes, 
            sortby='day_chrono_sort', 
            col_color=['day', 'cell_type'], 
            n_convolve=30,  
            figsize=(14, 8), 
            font_scale=0.7,  
            show=False
        )
        
        # Make x-axis gene labels perfectly flat/horizontal
        plt.xticks(rotation=0)
        
        # Fetch active canvas figure context to enforce Title properties directly
        fig = plt.gcf()
        fig.suptitle(f'{lin} Continuous Trajectory\n(20 Early & 20 Late Drivers Tracked across Days and Clusters)', 
                     y=1.12, fontsize=16, fontweight='bold')
        
        plt.show() 
        
    except Exception as e:
        print(f"  [!] Failed to generate heatmap for lineage '{lin}' due to error: {e}")

# Create the interactive widget
interact(plot_interactive_lineage_heatmap, lin=widgets.Dropdown(
    options=macro_lineages,
    value=macro_lineages[0] if len(macro_lineages) > 0 else None,
    description='Select Lineage:',
    disabled=False
))

print("\n--- Done ---")


--- INTERACTIVE: LATENT TIME DRIVERS PER MACRO-LINEAGE ---


interactive(children=(Dropdown(description='Select Lineage:', options=('Mesenchymal lineage', 'Myogenic lineag…


--- Done ---


**EOF**